# Multiple Linear Regression

## Data Loading & Exploration

In [ ]:
import pandas as pd
import numpy as np

from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn import metrics


def section(title):
    print("\n" + "=" * 60)
    print(title)
    print("=" * 60)


# Read CSV
df = pd.read_csv("../Day 11/data/advertising.csv")

section("LAST 5 ROWS")
print(df.tail())

section("SHAPE")
print("Shape:", df.shape)

section("COLUMNS")
print("Columns:", df.columns.tolist())

section("DATA TYPES & INFO")
print(df.info())

section("MISSING VALUES")
print(df.isnull().sum())

section("STATISTICAL SUMMARY")
print(df.describe())

section("DUPLICATE ROWS")
print("Duplicate rows:", df.duplicated().sum())


## Feature and Target Selection

In [ ]:
X = df[["TV", "Radio", "Newspaper"]]
y = df["Sales"]

print(X.shape)
print(y.shape)


## Train-Test Split

In [ ]:
X_train_bow, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print("Training set shape:", X_train_bow.shape, y_train.shape)
print("Testing set shape:", X_test.shape, y_test.shape)


## Model Training

In [ ]:
mr_model = LinearRegression()
mr_model.fit(X_train_bow, y_train)

print("Model coefficients:", mr_model.coef_)  # (Weights for TV, Radio, and Newspaper)
print("Model intercept:", mr_model.intercept_)  # Intercept of the regression line

## Overfitting Check (Train vs Test R²)

In [ ]:
# Overfitting check: Train and Test Scores
train_score = mr_model.score(X_train_bow, y_train)
test_score = mr_model.score(X_test, y_test)

print(f"Training Score (R^2): {train_score:.4f}")
print(f"Testing Score (R^2): {test_score:.4f}")


## RMSE Evaluation

In [ ]:
# Train RMSE
train_predictions = mr_model.predict(X_train_bow)
train_rmse = np.sqrt(metrics.mean_squared_error(y_train, train_predictions))

# Test RMSE
test_predictions = mr_model.predict(X_test)
test_rmse = np.sqrt(metrics.mean_squared_error(y_test, test_predictions))

print(f"Train RMSE: {train_rmse:.4f}")
print(f"Test RMSE: {test_rmse:.4f}")


## MAE Evaluation

In [ ]:
print(f"Train MAE: {metrics.mean_absolute_error(y_train, train_predictions):.4f}")
print(f"Test MAE: {metrics.mean_absolute_error(y_test, test_predictions):.4f}")

## R² Score Summary

In [ ]:
print(f"Train R2 Score: {mr_model.score(X_train_bow, y_train):.4f}")
print(f"Test R2 Score: {mr_model.score(X_test, y_test):.4f}")

## Overfitting Analysis: RMSE, MAE, R² Comparison

In [ ]:
# --- How to read RMSE/MAE/R2 to decide "overfitting or not" ---
# Overfitting = model memorizes training data instead of learning general patterns.
# Symptom: it does GREAT on train but MUCH WORSE on unseen test data.
# So the check is always the same shape: compare each metric's train value vs its test value.
#   - RMSE & MAE (error metrics, lower is better):
#       If Test RMSE/MAE >> Train RMSE/MAE -> errors balloon on unseen data -> overfitting.
#       If they're close -> model errs by roughly the same amount on both sets -> good generalization.
#   - R2 (goodness-of-fit, higher/closer to 1 is better):
#       If Train R2 >> Test R2 (big drop) -> model explains train variance well but fails on test -> overfitting.
#       If Train R2 ~= Test R2 (or test is even slightly higher) -> consistent performance -> no overfitting.
#       If both R2 scores are low -> that's UNDERFITTING instead (model too simple, not enough signal captured).

comparison = pd.DataFrame(
    {
        "Train": [
            train_rmse,
            metrics.mean_absolute_error(y_train, train_predictions),
            train_score,
        ],
        "Test": [
            test_rmse,
            metrics.mean_absolute_error(y_test, test_predictions),
            test_score,
        ],
    },
    index=["RMSE", "MAE", "R2"],
)
# The "Gap" column is the key diagnostic: a large positive gap on RMSE/MAE (test error much
# bigger than train error), or a large negative gap on R2 (test - train), signals overfitting.
comparison["Gap (Test - Train)"] = comparison["Test"] - comparison["Train"]

section("TRAIN VS TEST COMPARISON")
print(comparison)

# We use R2 as the deciding metric because it's on a fixed, interpretable scale (0 to 1),
# unlike RMSE/MAE which depend on the units of Sales. 0.05 is a rule-of-thumb threshold:
# a gap that small is normal randomness from the train/test split, not overfitting.
r2_gap = train_score - test_score  # positive -> train did better than test
if r2_gap > 0.05:
    # Train clearly outperforms test -> model learned train-specific noise -> overfitting.
    print(
        f"\nConclusion: R2 drops by {r2_gap:.4f} from train to test -> model is OVERFITTING."
    )
elif r2_gap < -0.05:
    # Test outperforming train by a lot is unusual -> often a sign of a small/unrepresentative
    # test split rather than a modeling problem -> worth investigating, not "overfitting".
    print(
        f"\nConclusion: Test R2 is higher than train R2 by {abs(r2_gap):.4f} -> unusual, check the split/data."
    )
else:
    # Gap is small in both directions, and RMSE/MAE tell the same story (see table above)
    # -> the model performs consistently on data it has and hasn't seen -> generalizes well.
    print(
        f"\nConclusion: RMSE, MAE, and R2 are close between train and test (R2 gap = {r2_gap:.4f}) -> model GENERALIZES WELL, no overfitting."
    )


## Prediction on New Data

In [ ]:
tv = 200
radio = 30
newspaper = 10

predictions_df = mr_model.predict(
    pd.DataFrame(
        {
            "TV": [tv],
            "Radio": [radio],
            "Newspaper": [newspaper],
        }
    )
)

print(
    f"Predicted Sales for TV={tv}, Radio={radio}, Newspaper={newspaper}: {predictions_df[0]:.4f}"
)

---

# Logistic Regression

In [ ]:
# Load the dataset for the classification part of this notebook:
# "Review Text" (raw customer review) as the feature, "sentiment" (-1/1) as
# the target -- this switches the task from regression (predicting a number)
# to classification (predicting a category).
df_reviews = pd.read_csv("data/womens_clothing_ecommerce_reviews.csv")
print(df_reviews.head())

## Data Quality Check (Empty, Null, Duplicates)

### Data Types & Missing Values

In [ ]:
section("DATA TYPES")
print("Dtype of each column:\n", df_reviews.dtypes)

# Null check: even though the text column is stored as a string dtype, a
# missing review can still show up as NaN (isnull() catches it; dtype alone
# does not).
section("MISSING (NULL) VALUES")
print(df_reviews.isnull().sum())
# Output: 0 nulls in both "Review Text" and "sentiment" -> no missing-value
# cleanup needed before modeling.

### Duplicate Rows

In [ ]:
# Duplicate reviews would let the same example influence training multiple
# times, effectively giving it more weight and biasing the model.
section("DUPLICATE ROWS")
print("Duplicate rows:", df_reviews.duplicated().sum())
# Output: 7 duplicate rows out of ~19.8k -> a tiny fraction, but worth
# dropping (df_reviews.drop_duplicates()) before training to avoid any bias.


df_reviews.drop_duplicates(inplace=True)
print("Duplicate rows after drop:", df_reviews.duplicated().sum())

### Empty Review Column

In [ ]:
# isnull() alone would miss a review that's an empty/whitespace-only string
# instead of true NaN, so check both explicitly.
section("EMPTY REVIEWS")
empty_reviews_count = (
    df_reviews["Review Text"].isnull().sum()
    + (df_reviews["Review Text"].str.strip() == "").sum()
)
print("Empty reviews:", empty_reviews_count)
# Output: 0 empty/whitespace-only reviews -> every row has real text to learn from.

## Class Balance - Counts

In [ ]:
# Logistic regression is a classifier: check how many reviews fall into each
# sentiment class. A skewed split (e.g. 90% positive) makes plain accuracy
# misleading, since always predicting the majority class would score high.
print(
    "Class counts (imbalance matters for accuracy):\n",
    df_reviews["sentiment"].value_counts(),
)
# Output: 17,448 positive (1) vs 2,370 negative (-1) -> the classes are
# heavily IMBALANCED (~88% / ~12%). A model that just predicts "positive"
# every time would score ~88% accuracy without learning anything -> plan to
# check precision/recall/F1 (or use class_weight="balanced") instead of
# relying on accuracy alone.

## Class Balance - Share (%)

In [ ]:
# Same check as above but as a percentage, easier to eyeball the imbalance ratio.
print("Class share")
print(df_reviews["sentiment"].value_counts(normalize=True).round(4))
# Output: ~88.04% positive vs ~11.96% negative -> confirms the imbalance seen
# in the raw counts above, roughly a 7:1 ratio of positive to negative reviews.

## Unique Label Values

In [ ]:
# Confirm the exact label encoding (e.g. -1/1 vs 0/1) before training, since
# sklearn's classifiers treat labels as class identifiers, not numbers to do
# arithmetic on.
print("Label values:", sorted(df_reviews["sentiment"].unique().tolist()))
# Output: [-1, 1] -> a clean binary label with no unexpected/stray values
# (e.g. no 0, no missing category), so no relabeling is needed.

## Feature and Target Selection

In [ ]:
# Same idea as the linear regression section: X = feature(s), y = target.
# The difference is that X here is raw TEXT, not numbers -- it can't be fed
# to a model as-is, which is why the next step (vectorization) exists.
X = df_reviews["Review Text"]
y = df_reviews["sentiment"]

print(X.shape)
print(y.shape)

## Train-Test Split

In [ ]:
# Split BEFORE vectorizing, not after: fitting the vectorizer on the whole
# dataset (train + test combined) would leak information about the test set's
# vocabulary into the model. Splitting the raw text first keeps the test set
# genuinely unseen until prediction time.
# stratify=y since the classes are imbalanced (~88%/12%, see Class Balance
# checks above) -- a plain random split could leave the test set with barely
# any negative reviews.
X_train_text, X_test_text, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print("Training set shape:", X_train_text.shape, y_train.shape)
print("Testing set shape:", X_test_text.shape, y_test.shape)
# Output: 15,848 training rows / 3,963 testing rows -> the 80/20 split is
# preserved, just like the linear regression section.

## Text Vectorization (CountVectorizer)

Why vectorize: TV/Radio/Newspaper were already numbers, but "Review Text" is raw text -- models only understand numbersCountVectorizer (bag-of-words) turns each review into a vector of raw word-occurrence counts: every word in the vocabulary becomes a column, and each cell is how many times that word appears in that review. This step has no equivalent in the linear regression section because that data was already numeric.

**How sentences become columns (a tiny worked example):**

Say the training text was just these 3 reviews (assume `stop_words="english"` has already dropped `"i"`, `"this"`, `"is"`):

| Review | Kept words |
|---|---|
| "I love this dress" | love, dress |
| "I hate this dress" | hate, dress |
| "This dress is amazing" | dress, amazing |

`fit` scans all 3 reviews and builds the **vocabulary** — every distinct word becomes its own column, in alphabetical order: `amazing, dress, hate, love`. That's 4 columns for 3 reviews.

`transform` then counts, for each review, how many times each vocabulary word appears — turning text into a numeric table:

| Review | amazing | dress | hate | love |
|---|---|---|---|---|
| "I love this dress" | 0 | 1 | 0 | 1 |
| "I hate this dress" | 0 | 1 | 1 | 0 |
| "This dress is amazing" | 1 | 1 | 0 | 0 |

Each row is exactly what `LogisticRegression` receives as one training example — a row of numbers instead of a sentence.

**Why this explains `fit_transform` vs `transform`:** the 4 columns above (`amazing, dress, hate, love`) only exist because `fit` was run on these 3 reviews. If a brand-new review came in later — say `"I love this amazing dress"` — `transform` reuses those *same 4 columns* (`[1, 1, 0, 1]`); it does **not** add a 5th column for any word it hasn't seen before, and it never re-learns the vocabulary. That's exactly what the code below does at real scale: `fit_transform(X_train_text)` builds a (much bigger, 5,000-word) vocabulary from the training reviews only, and `transform(X_test_text)` reuses those same 5,000 columns for the test reviews.

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer

vectorizer = CountVectorizer(stop_words="english", max_features=5000)

# Fit ONLY on the training text, so the vocabulary is learned purely from
# training data -- then just transform() the test text through that same
# vocabulary. Calling fit_transform on the test set (or on the full dataset
# before splitting) would leak test-set information into the model.
X_train_bow = vectorizer.fit_transform(X_train_text)
X_test = vectorizer.transform(X_test_text)

print("Training feature matrix shape:", X_train_bow.shape)
print("Testing feature matrix shape:", X_test.shape)
# Output: (15848, 5000) train / (3963, 5000) test -> same 5,000-word
# vocabulary applied to both, but learned from the training text only.

In [ ]:
print(X_train_bow[:5].toarray())  # Show the first 5 rows of the dense array representation

## Model Training

In [ ]:
# LogisticRegression instead of LinearRegression: it fits a decision boundary
# that outputs a class (-1 or 1) rather than a continuous number, by modeling
# the probability of each class with the logistic (sigmoid) function.
# max_iter=1000 gives the optimizer enough iterations to converge with 5,000
# features.
from sklearn.linear_model import LogisticRegression

log_model = LogisticRegression(max_iter=1000, random_state=42, solver="saga") # saga is a good choice for large datasets with many features, like our 5,000-word vocabulary.
log_model.fit(X_train_bow, y_train)

print("Model training complete! ✅")

print("Training accuracy:", log_model.score(X_train_bow, y_train))

print("Number of learned coefficients:", log_model.coef_.shape[1])
print("Intercept:", log_model.intercept_)
# Output: 5,000 coefficients (one weight per word) -- far too many to print
# individually like the 3 weights in the linear regression model, so we just
# report the count/intercept here.

## Overfitting Check (Train vs Test Accuracy)

In [ ]:
# Important distinction from the linear regression section: .score() on a
# classifier like LogisticRegression returns ACCURACY (% of correct
# predictions), not R^2 -- R^2 doesn't apply to classification. The
# overfitting logic is the same though: compare train vs test.
train_accuracy = log_model.score(X_train_bow, y_train)
test_accuracy = log_model.score(X_test, y_test)

print(f"Training Accuracy: {train_accuracy:.4f}")
print(f"Testing Accuracy: {test_accuracy:.4f}")
# Output: Train 0.9736 vs Test 0.9278 -> gap ~0.046. Still under the 0.05
# rule-of-thumb, but noticeably wider than with TF-IDF weighting -- raw word
# counts let the model fit the training vocabulary more tightly, pushing it
# closer to the overfitting boundary. Remember the imbalance warning from
# earlier too -- accuracy alone can hide a model that's bad at the minority
# class, which the classification report below confirms.

## Classification Metrics (Confusion Matrix & Report)

In [ ]:
# Why accuracy isn't enough here: ~88% of reviews are positive, so a model
# could score ~88% accuracy by ignoring negative reviews entirely. The
# confusion matrix and classification report show performance PER CLASS
# (precision, recall, F1) instead of one overall number.
from sklearn.metrics import classification_report, confusion_matrix

test_predictions = log_model.predict(X_test)

section("CONFUSION MATRIX")
print(confusion_matrix(y_test, test_predictions))

section("CLASSIFICATION REPORT")
print(classification_report(y_test, test_predictions, digits=4))
# Output: recall for the negative class (-1) is ~0.61 -- the model still
# catches meaningfully fewer negative reviews than positive ones (recall 0.61
# vs 0.97), even though overall accuracy looks strong at ~93%. This is the
# imbalance problem flagged earlier: accuracy alone hides how much worse the
# model does on the minority class.

## Prediction on New Data

In [ ]:
# New text must go through the SAME vectorizer.transform() (not fit_transform,
# which would relearn the vocabulary from this one sentence) before the model
# can use it -- mirroring how the linear regression prediction needed a
# DataFrame shaped like the training features.
new_review = ["The quality of this dress is amazing, I love it!"]
new_review_vectorized = vectorizer.transform(new_review)
prediction = log_model.predict(new_review_vectorized)

print(f"Review: {new_review[0]}")
print(f"Predicted Sentiment: {prediction[0]}")
# Output: Predicted Sentiment: 1 (positive) -- matches the clearly positive wording.